In [ ]:
import duckdb
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [74]:
con = duckdb.connect(r"C:\Users\jrf24\Incendios-Forestales-\data\BaseDeDatos_Working.db")

In [ ]:
con.execute("SHOW TABLES").df()

,name
0,causa
1,climatologia
2,danos
3,demografia
4,diccionario
5,estado
6,incendios
7,municipios
8,operaciones
9,vegetacion


In [76]:
# cargar tablas
df_incendios = con.execute("SELECT * FROM incendios").df()
df_clima = con.execute("SELECT * FROM climatologia").df()
df_causa = con.execute("SELECT * FROM causa").df()
df_veg = con.execute("SELECT * FROM vegetacion").df()
df_demo = con.execute("SELECT * FROM demografia").df()
df_danos = con.execute("SELECT * FROM danos").df()

In [77]:
df_clima.columns

Index(['id_registro', 'id_variable', 'id_clave_inc', 'fecha_de_observacion',
       'resultado_numerico'],
      dtype='str')

In [78]:
df_clima["fecha_de_observacion"] = pd.to_datetime(df_clima["fecha_de_observacion"])
df_incendios["fecha_inicio"] = pd.to_datetime(df_incendios["fecha_inicio"])

In [79]:
df_clima = df_clima.merge(
    df_incendios[["id_clave_inc", "fecha_inicio"]],
    on="id_clave_inc",
    how="left"
)

In [80]:
df_clima["dias_antes"] = (
    df_clima["fecha_inicio"] - df_clima["fecha_de_observacion"]
).dt.days

In [81]:
df_clima = df_clima[df_clima["dias_antes"] >= 0]

In [82]:
def crear_pivot(df, nombre, condicion):
    temp = df[condicion]
    
    pivot = temp.pivot_table(
        index="id_clave_inc",
        columns="id_variable",
        values="resultado_numerico",
        aggfunc="mean"
    )
    
    pivot.columns = [f"{col}_{nombre}" for col in pivot.columns]
    
    return pivot.reset_index()

In [83]:
pivot_0d = crear_pivot(df_clima, "0d", df_clima["dias_antes"] == 0)

pivot_3d = crear_pivot(df_clima, "3d", (df_clima["dias_antes"] > 0) & (df_clima["dias_antes"] <= 3))

pivot_7d = crear_pivot(df_clima, "7d", (df_clima["dias_antes"] > 0) & (df_clima["dias_antes"] <= 7))

pivot_15d = crear_pivot(df_clima, "15d", (df_clima["dias_antes"] > 0) & (df_clima["dias_antes"] <= 15))

In [84]:
df_clima_final = pivot_0d

df_clima_final = df_clima_final.merge(pivot_3d, on="id_clave_inc", how="left")
df_clima_final = df_clima_final.merge(pivot_7d, on="id_clave_inc", how="left")
df_clima_final = df_clima_final.merge(pivot_15d, on="id_clave_inc", how="left")

In [85]:
df = df_incendios.merge(df_clima, on="id_clave_inc", how="left")

In [86]:
df = df_incendios.merge(df_clima_final, on="id_clave_inc", how="left")

In [87]:
df = df.merge(df_veg, on="id_vegetacion", how="left")

In [88]:
df = df.merge(df_causa, on="id_causa", how="left")

In [89]:
df = df.merge(df_demo, on="id_cvegeo", how="left")

In [90]:
df = df.merge(df_danos, on="id_clave_inc", how="left")

In [91]:
df.head()

,id_clave_inc,id_cvegeo,id_causa,id_vegetacion,latitud,longitud,fecha_inicio,tipo_de_incendio,anio_x,AIRMASS_0d,...,r_70_74,r_75_79,r_80_84,r_85_mm,hojarasca,arbustivo,herbaceo,arbolado_adulto,renuevo,tamanio
0,15-15-0156,15039,32,28,19.309111,-99.788834,2015-03-02,superficial,2015,3.3,...,2573,1744,1130,915,0.0,0.0,15.0,0.0,0.0,11 a 20 hectareas
1,15-15-0156,15039,32,28,19.309111,-99.788834,2015-03-02,superficial,2015,3.3,...,3092,1847,1073,839,0.0,0.0,15.0,0.0,0.0,11 a 20 hectareas
2,15-15-0156,15039,32,28,19.309111,-99.788834,2015-03-02,superficial,2015,3.3,...,2620,1289,738,478,0.0,0.0,15.0,0.0,0.0,11 a 20 hectareas
3,15-15-0156,15039,32,28,19.309111,-99.788834,2015-03-02,superficial,2015,3.3,...,3092,1743,978,723,0.0,0.0,15.0,0.0,0.0,11 a 20 hectareas
4,15-15-0156,15039,32,28,19.309111,-99.788834,2015-03-02,superficial,2015,3.3,...,2869,1501,848,590,0.0,0.0,15.0,0.0,0.0,11 a 20 hectareas


In [92]:
df["temp_combustible"] = df["TSOIL6_0d"] + df["T2M_0d"]

df["estres_hidrico"] = df["NDVI_7d"] - df["NDMI_7d"]

In [93]:
df["presion_humana"] = df["pob_total"] / (
    df["latitud"].abs() + df["longitud"].abs() + 1
)

In [94]:
df["fecha_inicio"] = pd.to_datetime(df["fecha_inicio"])

df["mes"] = df["fecha_inicio"].dt.month
df["estacion"] = df["mes"] % 12 // 3 + 1

In [95]:
[col for col in df.columns if "NDVI" in col or "NDMI" in col or "TSOIL" in col]

['TSOIL1_0d',
 'TSOIL2_0d',
 'TSOIL3_0d',
 'TSOIL4_0d',
 'TSOIL5_0d',
 'TSOIL6_0d',
 'NDMI_3d',
 'NDVI_3d',
 'TSOIL1_3d',
 'TSOIL2_3d',
 'TSOIL3_3d',
 'TSOIL4_3d',
 'TSOIL5_3d',
 'TSOIL6_3d',
 'NDMI_7d',
 'NDVI_7d',
 'TSOIL1_7d',
 'TSOIL2_7d',
 'TSOIL3_7d',
 'TSOIL4_7d',
 'TSOIL5_7d',
 'TSOIL6_7d',
 'NDMI_15d',
 'NDVI_15d',
 'TSOIL1_15d',
 'TSOIL2_15d',
 'TSOIL3_15d',
 'TSOIL4_15d',
 'TSOIL5_15d',
 'TSOIL6_15d']

In [96]:
df["TSOIL_0d_mean"] = df[[f"TSOIL{i}_0d" for i in range(1,7)]].mean(axis=1)
df["TSOIL_3d_mean"] = df[[f"TSOIL{i}_3d" for i in range(1,7)]].mean(axis=1)
df["TSOIL_7d_mean"] = df[[f"TSOIL{i}_7d" for i in range(1,7)]].mean(axis=1)
df["TSOIL_15d_mean"] = df[[f"TSOIL{i}_15d" for i in range(1,7)]].mean(axis=1)

In [97]:
df["NDVI_mean"] = df[["NDVI_3d","NDVI_7d","NDVI_15d"]].mean(axis=1)

df["NDMI_mean"] = df[["NDMI_3d","NDMI_7d","NDMI_15d"]].mean(axis=1)

In [98]:
df["NDVI_trend"] = df["NDVI_3d"] - df["NDVI_15d"]

In [99]:
df["NDMI_trend"] = df["NDMI_3d"] - df["NDMI_15d"]

In [100]:
df["TSOIL_mean"] = df[
    ["TSOIL1_3d","TSOIL2_3d","TSOIL3_3d","TSOIL4_3d","TSOIL5_3d","TSOIL6_3d"]
].mean(axis=1)

In [101]:
df["TSOIL_trend"] = df["TSOIL6_3d"] - df["TSOIL6_15d"]

In [102]:
df["riesgo"] = pd.qcut(
    df["NDVI_trend"] - df["NDMI_trend"] + df["TSOIL_trend"],
    3,
    labels=["bajo", "medio", "alto"]
)

In [105]:
df.columns

Index(['id_clave_inc', 'id_cvegeo', 'id_causa', 'id_vegetacion', 'latitud',
       'longitud', 'fecha_inicio', 'tipo_de_incendio', 'anio_x', 'AIRMASS_0d',
       ...
       'TSOIL_3d_mean', 'TSOIL_7d_mean', 'TSOIL_15d_mean', 'NDVI_mean',
       'NDMI_mean', 'NDVI_trend', 'NDMI_trend', 'TSOIL_mean', 'TSOIL_trend',
       'riesgo'],
      dtype='str', length=541)

In [106]:
df_model = df.copy()

In [108]:
features = [
    # clima
    "TSOIL6", "NDVI", "NDMI", "EVPTRNS", "T2M",

    # vegetación
    "id_vegetacion", "estres_hidrico", "temp_combustible",

    # demografía
    "poblacion_total", "presion_humana",

    # espacial / temporal
    "latitud", "longitud", "anio", "mes", "estacion"
]

In [111]:
print(df_model.columns.tolist())

['id_clave_inc', 'id_cvegeo', 'id_causa', 'id_vegetacion', 'latitud', 'longitud', 'fecha_inicio', 'tipo_de_incendio', 'anio_x', 'AIRMASS_0d', 'ALLSKY_KT_0d', 'ALLSKY_NKT_0d', 'ALLSKY_SFC_LW_DWN_0d', 'ALLSKY_SFC_LW_UP_0d', 'ALLSKY_SFC_PAR_DIFF_0d', 'ALLSKY_SFC_PAR_DIRH_0d', 'ALLSKY_SFC_PAR_TOT_0d', 'ALLSKY_SFC_SW_DIFF_0d', 'ALLSKY_SFC_SW_DIRH_0d', 'ALLSKY_SFC_SW_DNI_0d', 'ALLSKY_SFC_SW_DWN_0d', 'ALLSKY_SFC_SW_UP_0d', 'ALLSKY_SFC_UVA_0d', 'ALLSKY_SFC_UVB_0d', 'ALLSKY_SFC_UV_INDEX_0d', 'ALLSKY_SRF_ALB_0d', 'AOD_55_0d', 'AOD_55_ADJ_0d', 'AOD_84_0d', 'CDD0_0d', 'CDD18_3_0d', 'CLOUD_AMT_0d', 'CLOUD_AMT_DAY_0d', 'CLOUD_AMT_NIGHT_0d', 'CLOUD_OD_0d', 'CLRSKY_DAYS_0d', 'CLRSKY_KT_0d', 'CLRSKY_NKT_0d', 'CLRSKY_SFC_LW_DWN_0d', 'CLRSKY_SFC_LW_UP_0d', 'CLRSKY_SFC_PAR_DIFF_0d', 'CLRSKY_SFC_PAR_DIRH_0d', 'CLRSKY_SFC_PAR_TOT_0d', 'CLRSKY_SFC_SW_DIFF_0d', 'CLRSKY_SFC_SW_DIRH_0d', 'CLRSKY_SFC_SW_DNI_0d', 'CLRSKY_SFC_SW_DWN_0d', 'CLRSKY_SFC_SW_UP_0d', 'CLRSKY_SRF_ALB_0d', 'DISPH_0d', 'EVLAND_0d', 'EVPTRNS

In [112]:
df = df.copy()

In [113]:
features = [
    # Vegetación
    "NDVI_mean",
    "NDMI_mean",
    "NDVI_trend",
    "NDMI_trend",

    # Suelo/clima resumido
    "TSOIL_mean",
    "TSOIL_trend",

    # Demografía
    "pob_total",
    "presion_humana",

    # incendios/estructura
    "mes",
    "estacion",
    "id_vegetacion",
    "id_causa"
]

df_model = df[features + ["riesgo"]].dropna()

In [114]:
X = df_model[features]
y = df_model["riesgo"]

In [115]:
from sklearn.ensemble import RandomForestClassifier

model = RandomForestClassifier(
    n_estimators=150,
    max_depth=12,
    n_jobs=-1,
    random_state=42
)

model.fit(X, y)

,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",150
,"criterion criterion: {""gini"", ""entropy"", ""log_loss""}, default=""gini""The function to measure the quality of a split. Supported criteria are""gini"" for the Gini impurity and ""log_loss"" and ""entropy"" both for theShannon information gain, see :ref:`tree_mathematical_formulation`.Note: This parameter is tree-specific.",'gini'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",12
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: {""sqrt"", ""log2"", None}, int or float, default=""sqrt""The number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None, then `max_features=n_features`... versionchanged:: 1.1 The default of `max_features` changed from `""auto""` to `""sqrt""`.Note: the search for a split does not stop until at least onevalid partition of the node samples is found, even if it requires toeffectively inspect more than ``max_features`` features.",'sqrt'
,"max_leaf_nodes max_leaf_nodes: int, default=NoneGrow trees with ``max_leaf_nodes`` in best-first fashion.Best nodes are defined as relative reduction in impurity.If None then unlimited number of leaf nodes.",None
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease of the impuritygreater than or equal to this value.The weighted impurity decrease equation is the following:: N_t / N * (impurity - N_t_R / N_t * right_impurity - N_t_L / N_t * left_impurity)where ``N`` is the total number of samples, ``N_t`` is the number ofsamples at the current node, ``N_t_L`` is the number of samples in theleft child, and ``N_t_R`` is the number of samples in the right child.``N``, ``N_t``, ``N_t_R`` and ``N_t_L`` all refer to the weighted sum,if ``sample_weight`` is passed... versionadded:: 0.19",0.0
,"bootstrap bootstrap: bool, default=TrueWhether bootstrap samples are used when building trees. If False, thewhole dataset is used to build each tree.",True
,"oob_score oob_score: bool or callable, default=FalseWhether to use out-of-bag samples to estimate the generalization score.By default, :func:`~sklearn.metrics.accuracy_score` is used.Provide a callable with signature `metric(y

In [116]:
import pandas as pd

importancias = pd.Series(
    model.feature_importances_,
    index=X.columns
).sort_values(ascending=False)

importancias

TSOIL_trend       0.839780
TSOIL_mean        0.070092
mes               0.049408
estacion          0.012463
pob_total         0.006017
NDMI_mean         0.005623
NDVI_mean         0.005246
presion_humana    0.005137
id_causa          0.003239
id_vegetacion     0.002995
NDMI_trend        0.000000
NDVI_trend        0.000000
dtype: float64

In [119]:
importancias.head(20)

TSOIL_trend       0.839780
TSOIL_mean        0.070092
mes               0.049408
estacion          0.012463
pob_total         0.006017
NDMI_mean         0.005623
NDVI_mean         0.005246
presion_humana    0.005137
id_causa          0.003239
id_vegetacion     0.002995
NDMI_trend        0.000000
NDVI_trend        0.000000
dtype: float64

In [120]:
RandomForestClassifier(
    max_depth=20,
    min_samples_leaf=5,
    class_weight="balanced"
)

,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",100
,"criterion criterion: {""gini"", ""entropy"", ""log_loss""}, default=""gini""The function to measure the quality of a split. Supported criteria are""gini"" for the Gini impurity and ""log_loss"" and ""entropy"" both for theShannon information gain, see :ref:`tree_mathematical_formulation`.Note: This parameter is tree-specific.",'gini'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",20
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",5
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: {""sqrt"", ""log2"", None}, int or float, default=""sqrt""The number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None, then `max_features=n_features`... versionchanged:: 1.1 The default of `max_features` changed from `""auto""` to `""sqrt""`.Note: the search for a split does not stop until at least onevalid partition of the node samples is found, even if it requires toeffectively inspect more than ``max_features`` features.",'sqrt'
,"max_leaf_nodes max_leaf_nodes: int, default=NoneGrow trees with ``max_leaf_nodes`` in best-first fashion.Best nodes are defined as relative reduction in impurity.If None then unlimited number of leaf nodes.",None
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease of the impuritygreater than or equal to this value.The weighted impurity decrease equation is the following:: N_t / N * (impurity - N_t_R / N_t * right_impurity - N_t_L / N_t * left_impurity)where ``N`` is the total number of samples, ``N_t`` is the number ofsamples at the current node, ``N_t_L`` is the number of samples in theleft child, and ``N_t_R`` is the number of samples in the right child.``N``, ``N_t``, ``N_t_R`` and ``N_t_L`` all refer to the weighted sum,if ``sample_weight`` is passed... versionadded:: 0.19",0.0
,"bootstrap bootstrap: bool, default=TrueWhether bootstrap samples are used when building trees. If False, thewhole dataset is used to build each tree.",True
,"oob_score oob_score: bool or callable, default=FalseWhether to use out-of-bag samples to estimate the generalization score.By default, :func:`~sklearn.metrics.accuracy_score` is used.Provide a callable with signature `metric(y

In [122]:
import pandas as pd

importancias = pd.Series(
    model.feature_importances_,
    index=X.columns
).sort_values(ascending=False)

importancias

TSOIL_trend       0.839780
TSOIL_mean        0.070092
mes               0.049408
estacion          0.012463
pob_total         0.006017
NDMI_mean         0.005623
NDVI_mean         0.005246
presion_humana    0.005137
id_causa          0.003239
id_vegetacion     0.002995
NDMI_trend        0.000000
NDVI_trend        0.000000
dtype: float64

In [121]:
df_model = df_model.drop(columns=["TSOIL_trend"])

In [123]:
import pandas as pd

importancias = pd.Series(
    model.feature_importances_,
    index=X.columns
).sort_values(ascending=False)

importancias

TSOIL_trend       0.839780
TSOIL_mean        0.070092
mes               0.049408
estacion          0.012463
pob_total         0.006017
NDMI_mean         0.005623
NDVI_mean         0.005246
presion_humana    0.005137
id_causa          0.003239
id_vegetacion     0.002995
NDMI_trend        0.000000
NDVI_trend        0.000000
dtype: float64

In [124]:
clima_vars = [c for c in df.columns if any(x in c for x in [
    "TSOIL", "T2M", "T10M", "EVPTRNS", "NDMI", "NDVI",
    "PREC", "RH2M", "WS10M", "SLP", "AIRMASS"
])]

In [125]:
vegetacion_vars = [
    "id_vegetacion",
    "NDVI_mean", "NDMI_mean",
    "NDVI_trend", "NDMI_trend",
    "TSOIL_mean", "TSOIL_trend",
    "hojarasca", "arbustivo", "herbaceo",
    "arbolado_adulto", "renuevo"
]

In [126]:
humano_vars = [
    "pob_total",
    "presion_humana",
    "id_causa",
    "mes",
    "estacion"
]

In [127]:
from sklearn.ensemble import RandomForestClassifier

model = RandomForestClassifier(
    n_estimators=200,
    random_state=42,
    n_jobs=-1
)

model.fit(X, y)

,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",200
,"criterion criterion: {""gini"", ""entropy"", ""log_loss""}, default=""gini""The function to measure the quality of a split. Supported criteria are""gini"" for the Gini impurity and ""log_loss"" and ""entropy"" both for theShannon information gain, see :ref:`tree_mathematical_formulation`.Note: This parameter is tree-specific.",'gini'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",None
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: {""sqrt"", ""log2"", None}, int or float, default=""sqrt""The number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None, then `max_features=n_features`... versionchanged:: 1.1 The default of `max_features` changed from `""auto""` to `""sqrt""`.Note: the search for a split does not stop until at least onevalid partition of the node samples is found, even if it requires toeffectively inspect more than ``max_features`` features.",'sqrt'
,"max_leaf_nodes max_leaf_nodes: int, default=NoneGrow trees with ``max_leaf_nodes`` in best-first fashion.Best nodes are defined as relative reduction in impurity.If None then unlimited number of leaf nodes.",None
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease of the impuritygreater than or equal to this value.The weighted impurity decrease equation is the following:: N_t / N * (impurity - N_t_R / N_t * right_impurity - N_t_L / N_t * left_impurity)where ``N`` is the total number of samples, ``N_t`` is the number ofsamples at the current node, ``N_t_L`` is the number of samples in theleft child, and ``N_t_R`` is the number of samples in the right child.``N``, ``N_t``, ``N_t_R`` and ``N_t_L`` all refer to the weighted sum,if ``sample_weight`` is passed... versionadded:: 0.19",0.0
,"bootstrap bootstrap: bool, default=TrueWhether bootstrap samples are used when building trees. If False, thewhole dataset is used to build each tree.",True
,"oob_score oob_score: bool or callable, default=FalseWhether to use out-of-bag samples to estimate the generalization score.By default, :func:`~sklearn.metrics.accuracy_score` is used.Provide a callable with signature `metric

In [128]:
import pandas as pd

importancias = pd.Series(
    model.feature_importances_,
    index=X.columns
).sort_values(ascending=False)

In [129]:
def importance_group(group_vars, importancias):
    return importancias.loc[
        [v for v in group_vars if v in importancias.index]
    ].sum()

In [130]:
score_clima = importance_group(clima_vars, importancias)
score_veg = importance_group(vegetacion_vars, importancias)
score_humano = importance_group(humano_vars, importancias)

In [131]:
pd.Series({
    "Clima": score_clima,
    "Vegetación": score_veg,
    "Humano": score_humano
}).sort_values(ascending=False)

Vegetación    0.922743
Clima         0.919055
Humano        0.077257
dtype: float64